In [316]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import os
import itertools
from sklearn.metrics.pairwise import cosine_similarity

In [317]:
topic_category = {
    '文化与旅游': 'Culture',
    '医疗与公共健康': 'Health',
    '政治与政策': 'Politics',
    '民生与便民服务': 'Livelihood',
    '经济与金融': 'Economy',
    '就业与培训': 'Employment',
    '交通与基础设施': 'Transport',
    '应急与安全': 'Safety',
    '教育与人才': 'Edu & Talent',
    '法律与治安管理': 'Law',
    '环境与可持续发展': 'Environment',
    '体育赛事': 'Sports',
    '科技与创新': 'Science',
    '其他': 'Others'
}

In [318]:
all_labels = [x for x in topic_category.keys() if x != '其他']

def get_cooccurrence_matrix(df, label_col='label', sep='@', all_labels=all_labels, keep_diag=True):
    data = df.copy()
    # 拆分标签
    data['_label_list'] = data[label_col].fillna('').apply(
        lambda x: [i.strip() for i in str(x).split(sep) if i.strip()]
    )
    # 每行去重，避免同一行重复标签重复计数
    data['_label_list'] = data['_label_list'].apply(lambda x: list(dict.fromkeys(x)))
    # 如果没传 all_labels，就从数据中自动收集
    if all_labels is None:
        all_labels = sorted(set(itertools.chain.from_iterable(data['_label_list'])))
    # 初始化矩阵
    mat = pd.DataFrame(0, index=all_labels, columns=all_labels, dtype=int)
    # 统计共现
    for labels in data['_label_list']:
        # 若传入了 all_labels，则过滤非法标签
        labels = [x for x in labels if x in all_labels]

        if keep_diag:
            for lab in labels:
                mat.loc[lab, lab] += 1

        for a, b in itertools.combinations(labels, 2):
            mat.loc[a, b] += 1
            mat.loc[b, a] += 1

    # 保证所有类别都在矩阵里
    mat = mat.reindex(index=all_labels, columns=all_labels, fill_value=0)
    # 如果不要对角线，统一置 0
    if not keep_diag:
        for lab in all_labels:
            mat.loc[lab, lab] = 0

    co_mat_en = mat.copy()
    co_mat_en.index = [topic_category[x] for x in co_mat_en.index]
    co_mat_en.columns = [topic_category[x] for x in co_mat_en.columns]

    return co_mat_en

In [320]:
def compute_network_centralities(matrix, labels=None, as_distance=True):
    # 1.转成 DataFrame
    if isinstance(matrix, pd.DataFrame):
        df = matrix.copy()
        if labels is None:
            labels = list(df.index)
    else:
        matrix = np.array(matrix)
        n = matrix.shape[0]
        if labels is None:
            labels = [f"Node_{i}" for i in range(n)]
        df = pd.DataFrame(matrix, index=labels, columns=labels)

    # 2.cheeck matrix
    if df.shape[0] != df.shape[1]:
        raise ValueError("matrix 必须是方阵。")
    if len(df.index) != len(df.columns):
        raise ValueError("matrix 的行列维度不一致。")

    # 3.构建无向加权图
    G = nx.from_pandas_adjacency(df)

    # 4.带权度中心性
    weighted_degree = dict(G.degree(weight="weight"))

    # 5.为 betweenness / closeness 构造 distance
    # weight 表示“关系强度”， shortest path = distance = 1 / weight
    H = G.copy()
    for u, v, d in H.edges(data=True):
        w = d.get("weight", 0)
        if w <= 0:
            # 避免除零
            dist = np.inf
        else:
            dist = 1 / w if as_distance else w

        d["distance"] = dist

    # 6.介数中心性
    betweenness = nx.betweenness_centrality(H, weight="distance", normalized=True)

    # 7.接近中心性
    closeness = nx.closeness_centrality(H, distance="distance")

    # 8.特征向量中心性
    try:
        eigenvector = nx.eigenvector_centrality_numpy(G, weight="weight")
    except Exception:
        eigenvector = nx.eigenvector_centrality(G, weight="weight", max_iter=1000)

    # 9.结果
    result_df = pd.DataFrame({
        "weighted_degree": pd.Series(weighted_degree),
        "betweenness_centrality": pd.Series(betweenness),
        "closeness_centrality": pd.Series(closeness),
        "eigenvector_centrality": pd.Series(eigenvector),
    })

    return result_df

In [321]:
local_df = pd.read_csv('./data/topic-results/result_local_topic.csv')[['idx', 'label']]
local_oc = get_cooccurrence_matrix(local_df, keep_diag=False)
local_oc.reset_index().to_excel('./analysis-data/local_topic-co_matrix.xlsx', index=None)

In [322]:
compute_network_centralities(local_oc)

,weighted_degree,betweenness_centrality,closeness_centrality,eigenvector_centrality
Culture,657,0.166667,42.514464,0.451139
Health,346,0.000000,30.855443,0.210260
Politics,505,0.000000,37.914033,0.338198
Livelihood,580,0.181818,40.624541,0.333686
Economy,867,0.500000,47.454439,0.530686
Employment,307,0.000000,35.379596,0.223845
Transport,409,0.060606,36.746036,0.252737
Safety,170,0.000000,18.349031,0.083466
Edu & Talent,359,0.000000,29.735434,0.232395
Law,177,0.000000,18.843007,0.093355


In [323]:
non_df = pd.read_csv('./data/topic-results/result_non_local.csv')
non_oc = get_cooccurrence_matrix(non_df, keep_diag=False)
non_oc.reset_index().to_excel('./analysis-data/non_topic-co_matrix.xlsx', index=None)

In [324]:
compute_network_centralities(non_oc)

,weighted_degree,betweenness_centrality,closeness_centrality,eigenvector_centrality
Culture,841,0.166667,36.884947,0.409073
Health,369,0.000000,28.175958,0.169023
Politics,944,0.106061,37.591246,0.470014
Livelihood,742,0.030303,35.094682,0.331481
Economy,1221,0.545455,40.226859,0.534867
Employment,210,0.000000,24.213939,0.101702
Transport,787,0.393939,37.712222,0.346573
Safety,26,0.000000,9.103328,0.011077
Edu & Talent,436,0.000000,29.795903,0.192920
Law,142,0.000000,18.898057,0.062435


In [325]:
def output_freq(df):
    cols = list(topic_category.keys())
    data = []
    for cl in cols:
        data.append([cl, len(df[df.label.str.contains(cl)])])
    t = pd.DataFrame(data, columns=['category', 'count'])
    t = t.sort_values(by='count', ascending=False)
    return t

In [326]:
output_freq(local_df)

,category,count
0,文化与旅游,665
4,经济与金融,593
1,医疗与公共健康,545
3,民生与便民服务,403
2,政治与政策,308
6,交通与基础设施,207
5,就业与培训,193
8,教育与人才,154
7,应急与安全,123
9,法律与治安管理,122


In [327]:
output_freq(non_df)

,category,count
2,政治与政策,1821
0,文化与旅游,993
4,经济与金融,904
6,交通与基础设施,624
3,民生与便民服务,537
1,医疗与公共健康,417
8,教育与人才,343
13,其他,231
12,科技与创新,143
9,法律与治安管理,142


## 按照城市构造主题矩阵 => 标准化三角（考虑到主题频次和共现结构，更新！）

In [328]:
def get_upper_values(df):
    MM = get_cooccurrence_matrix(df, keep_diag=True)
    news_count = len(df)

    upper_idx = np.triu_indices_from(MM, k=0)  # 包含对角线
    MM_list = MM.values[upper_idx].tolist()

    res_data = [i / news_count for i in MM_list]
    return res_data

### 省份

In [329]:
def output_all_provinces_co_occ_matrix():
    pros = non_df['province'].drop_duplicates().tolist()
    print(len(pros))
    all_result = []

    for province in pros:
        temp = non_df[non_df['province']==province]
        res_data = get_upper_values(temp)
        all_result.append([province, res_data])
    return pd.DataFrame(all_result, columns=['province', 'martix_upper'])

df_province = output_all_provinces_co_occ_matrix()

31


In [330]:
df_Macau = pd.DataFrame([['澳门', get_upper_values(local_df)]], columns=['province', 'martix_upper'])
df_ALL_PROS = pd.concat([df_Macau, df_province]).reset_index(drop=True)

In [331]:
# 计算余弦相似度
def get_cosine_similarity_matrix(df, feature_col='martix_upper', name_col='province'):
    data = df.copy()
    X = np.array(data[feature_col].tolist())
    names = data[name_col].tolist()

    sim_matrix = cosine_similarity(X)

    sim_df = pd.DataFrame(sim_matrix, index=names, columns=names)
    return sim_df

### 城市

In [333]:
def output_all_cities_co_occ_matrix():
    cities = non_df['city_name'].drop_duplicates().tolist()
    print(len(cities))
    all_result = []

    for city in cities:
        temp = non_df[non_df['city_name']==city]
        res_data = get_upper_values(temp)
        all_result.append([city, res_data])
    return pd.DataFrame(all_result, columns=['city', 'martix_upper'])

df_cities = output_all_cities_co_occ_matrix()
df_Macau_as_city = pd.DataFrame([['澳门', get_upper_values(local_df)]], columns=['city', 'martix_upper'])
df_ALL_CITIES = pd.concat([df_Macau_as_city, df_cities]).reset_index(drop=True)

258


In [334]:
cos_sim_df_CITIES = get_cosine_similarity_matrix(df_ALL_CITIES, feature_col='martix_upper', name_col='city')
cos_sim_CITIES = cos_sim_df_CITIES.iloc[0].sort_values(ascending=False).mul(100).round(2)
CITIES_SIM = cos_sim_CITIES.reset_index()
CITIES_SIM.columns = ['name', 'similarity']
city_profile = pd.read_csv('./data/city_profile.csv')
CITIES_SIM = pd.merge(city_profile, CITIES_SIM, on='name')
CITIES_SIM = CITIES_SIM.sort_values(by='similarity', ascending=False)
CITIES_SIM.to_csv('./analysis-data/macau_other_cities_sim.csv', index=None)
CITIES_SIM.to_excel('./analysis-data/macau_other_cities_sim.xlsx', index=None)
CITIES_SIM

,name,center,pro_name,pro_shortname,en_name,similarity
159,佛山市,"113.121586,23.021351",广东省,广东,Foshan,91.94
163,珠海市,"113.576892,22.271644",广东省,广东,Zhuhai,89.88
176,中山市,"113.392517,22.517024",广东省,广东,Zhongshan,87.12
177,东莞市,"113.751884,23.021016",广东省,广东,Dongguan,87.10
162,深圳市,"114.057939,22.543527",广东省,广东,Shenzhen,87.03
...,...,...,...,...,...,...
116,泰安市,"117.086963,36.201784",山东省,山东,Tai'an,23.74
253,哈密市,"93.515053,42.819346",新疆维吾尔自治区,新疆,Hami,11.87
216,保山市,"99.161489,25.112018",云南省,云南,Baoshan,11.87
87,六安市,"116.519729,31.735892",安徽省,安徽,Lu'an,0.00


In [343]:
line_chart = CITIES_SIM.copy()
line_chart['similarity_x'] = list(range(len(line_chart)))
line_chart['similarity_y'] = line_chart['similarity'].map(lambda x: f"{x:.2f}")
line_chart[['similarity_x', 'similarity_y']].to_excel('./analysis-data/city_sim_line_chart.xlsx', index=None)

In [344]:
line_chart[['similarity_x', 'similarity_y']]

,similarity_x,similarity_y
159,0,91.94
163,1,89.88
176,2,87.12
177,3,87.10
162,4,87.03
...,...,...
116,253,23.74
253,254,11.87
216,255,11.87
87,256,0.00


In [349]:
line_chart['similarity'].describe()

count    258.000000
mean      53.240465
std       17.896708
min        0.000000
25%       40.007500
50%       52.955000
75%       68.615000
max       91.940000
Name: similarity, dtype: float64

In [346]:
CITIES_SIM.to_csv('./QGIS/similarity_data.csv', index=None)

In [350]:
# 计算占比
def count_similarity_bins(df, col='similarity'):
    bins = [0, 25, 50, 75, 100.000001]
    labels = ['0-25', '25-50', '50-75', '75-100']
    
    result = pd.cut(
        df[col],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    ).value_counts().sort_index()
    
    return result

In [351]:
count_similarity_bins(line_chart)

similarity
0-25       21
25-50      92
50-75     113
75-100     32
Name: count, dtype: int64

In [352]:
21 + 92 + 113 +32

258